In [ ]:
import numpy as np

def mat_mult(mat1, mat2):
    r1 = len(mat1)
    c1 = len(mat1[0])
    r2 = len(mat2)
    c2 = len(mat2[0])

    if c1 != r2:
        raise ValueError("The matrices cannot be multiplied: inner dimensions differ.")

    result = np.zeros((r1, c2))
    for i in range(r1):
        for j in range(c2):
            total = 0.0
            for k in range(c1):
                total += mat1[i, k] * mat2[k, j]
            result[i, j] = total
    return result


def matrix_transpose(mat):
    rows = len(mat)
    cols = len(mat[0])
    transposed = np.zeros((cols, rows))
    for i in range(rows):
        for j in range(cols):
            transposed[j, i] = mat[i, j]
    return transposed


def gaussian_elimination(aug_mat, tolerance=1e-12):
    ref_mat = aug_mat.copy().astype(float)
    n_rows, n_cols = ref_mat.shape
    coeff_cols = n_cols - 1
    pivot_row = 0

    for col in range(coeff_cols):
        if pivot_row == n_rows:
            break

        candidate_row = pivot_row
        while (
            candidate_row < n_rows
            and abs(ref_mat[candidate_row, col]) < tolerance
        ):
            candidate_row += 1

        if candidate_row == n_rows:
            continue

        ref_mat[[pivot_row, candidate_row]] = ref_mat[[candidate_row, pivot_row]]

        ref_mat[pivot_row] = ref_mat[pivot_row] / ref_mat[pivot_row, col]

        for next_row in range(pivot_row + 1, n_rows):
            mult = ref_mat[next_row, col]
            ref_mat[next_row] = (
                ref_mat[next_row] - mult * ref_mat[pivot_row]
            )

        pivot_row += 1

    return ref_mat


def reduced_row_form(aug_mat, tolerance=1e-12):
    rref_mat = gaussian_elimination(aug_mat, tolerance)
    n_rows, n_cols = rref_mat.shape
    coeff_cols = n_cols - 1

    for pivot_row in range(n_rows - 1, -1, -1):
        pivot_col = 0
        while (
            pivot_col < coeff_cols
            and abs(rref_mat[pivot_row, pivot_col]) < tolerance
        ):
            pivot_col += 1

        if pivot_col == coeff_cols:
            continue

        for prev_row in range(pivot_row):
            mult = rref_mat[prev_row, pivot_col]
            rref_mat[prev_row] = (
                rref_mat[prev_row] - mult * rref_mat[pivot_row]
            )

    return rref_mat


while True:
    print("Input matrix dimensions - A = m x n, where m < n")
    rows = int(input("m = "))
    cols = int(input("n = "))
    if rows > 0 and cols > 0 and rows < cols:
        break
    print("Invalid input. Please enter positive integers with m < n.")

np.random.seed(1991)
matrix_A = np.random.uniform(1.5, 8.5, size=(rows, cols))
vector_b = np.random.uniform(1.5, 8.5, size=(rows, 1))
aug_matrix = np.hstack((matrix_A, vector_b))

print("\nMatrix A =")
print(matrix_A)
print("\nVector b =")
print(vector_b)
print("\nAugmented matrix [A | b] =")
print(aug_matrix)

ref = gaussian_elimination(aug_matrix)
rref = reduced_row_form(aug_matrix)

print("\nREF =")
print(ref)
print("\nRREF =")
print(rref)

In [ ]:
def check_row_consistency(rref_mat, coeff_cols, tolerance=1e-12):
    n_rows = len(rref_mat)
    for row in range(n_rows):
        all_zero = True

        for col in range(coeff_cols):
            if abs(rref_mat[row, col]) >= tolerance:
                all_zero = False
                break

        if (
            all_zero
            and abs(rref_mat[row, coeff_cols]) >= tolerance
        ):
            return True

    return False


rref = reduced_row_form(aug_matrix)

if check_row_consistency(rref, cols):
    print("The system is inconsistent and has no solution.")
    raise SystemExit("Program stopped because no solution exists.")

pivot_cols = []
for row in range(rows):
    for col in range(cols):
        if abs(rref[row, col]) > 1e-12:
            pivot_cols.append(col)
            break

free_cols = []
for col in range(cols):
    if col not in pivot_cols:
        free_cols.append(col)

print("Pivot columns (0-based):", pivot_cols)
print("Non-pivot columns (0-based):", free_cols)

particular_sol = np.zeros(cols)
for row, pivot_col in enumerate(pivot_cols):
    particular_sol[pivot_col] = rref[row, cols]

print("\nParticular solution x_p =")
print(particular_sol)
print("Check A x_p =")
print(mat_mult(matrix_A, particular_sol.reshape(cols, 1)).reshape(rows))
print("Check b =")
print(vector_b.reshape(rows))

null_vecs = []
for free_col in free_cols:
    vec = np.zeros(cols)
    vec[free_col] = 1

    for row, pivot_col in enumerate(pivot_cols):
        vec[pivot_col] = -rref[row, free_col]
    null_vecs.append(vec)

print("\nVectors for solutions of A x = 0:")
for number, vec in enumerate(null_vecs, start=1):
    print("v" + str(number) + " =", vec)
    print("A v" + str(number) + " =")
    print(mat_mult(matrix_A, vec.reshape(cols, 1)).reshape(rows))

print("\nGeneral solution:")
print("x = x_p + c1*v1 + c2*v2 + ...")

In [ ]:
np.random.seed(1991)
mat_4x6 = np.random.uniform(1.5, 8.5, size=(4, 6))
vec_4x6 = np.random.uniform(1.5, 8.5, size=(4, 1))
aug_4x6 = np.hstack((mat_4x6, vec_4x6))

ref_4x6 = gaussian_elimination(aug_4x6)
rref_4x6 = reduced_row_form(aug_4x6)

print("A =")
print(mat_4x6)
print("\nb =")
print(vec_4x6.reshape(4))
print("\nREF of [A | b] =")
print(ref_4x6)
print("\nRREF of [A | b] =")
print(rref_4x6)

if check_row_consistency(rref_4x6, 6):
    print("\nThe 4 x 6 system is inconsistent and has no solution.")
    raise SystemExit("Program stopped because no solution exists.")

pivot_cols_4x6 = []
for row in range(4):
    for col in range(6):
        if abs(rref_4x6[row, col]) > 1e-12:
            pivot_cols_4x6.append(col)
            break

free_cols_4x6 = [
    col for col in range(6) if col not in pivot_cols_4x6
]

print("\nPivot columns:", pivot_cols_4x6)
print("Non-pivot columns:", free_cols_4x6)

particular_sol_4x6 = np.zeros(6)
for row, pivot_col in enumerate(pivot_cols_4x6):
    particular_sol_4x6[pivot_col] = rref_4x6[row, 6]

print("\nParticular solution x_p =")
print(particular_sol_4x6)
print("Check A x_p =")
print(mat_mult(mat_4x6, particular_sol_4x6.reshape(6, 1)).reshape(4))
print("Check b =")
print(vec_4x6.reshape(4))

null_vecs_4x6 = []
for free_col in free_cols_4x6:
    vec = np.zeros(6)
    vec[free_col] = 1
    for row, pivot_col in enumerate(pivot_cols_4x6):
        vec[pivot_col] = -rref_4x6[row, free_col]
    null_vecs_4x6.append(vec)

print("\nSolutions to A x = 0:")
for number, vec in enumerate(null_vecs_4x6, start=1):
    print("v" + str(number) + " =", vec)
    print("A v" + str(number) + " =")
    print(mat_mult(mat_4x6, vec.reshape(6, 1)).reshape(4))

print("\nGeneral solution:")
print("x = x_p + c1*v1 + c2*v2 + ...")
print("The checks above verify A x_p = b and A v_i = 0.")

In [ ]:
import numpy as np

np.random.seed(1991)
sample_count = 350

feat_a = np.random.randn(sample_count)
feat_b = np.random.randn(sample_count)
feat_c = np.random.randn(sample_count)
feat_d = np.random.randn(sample_count)

feat_e = 3 * feat_a + 2 * feat_b
feat_f = 2 * feat_c - feat_d

dataset = np.column_stack((feat_a, feat_b, feat_c, feat_d, feat_e, feat_f))

print("Shape of dataset:", dataset.shape)
print("First five rows:")
print(dataset[:5])

In [ ]:
reduced_data = dataset.copy().astype(float)
n_rows, n_cols = reduced_data.shape
matrix_rank = 0
tolerance = 1e-12

for col in range(n_cols):
    if matrix_rank == n_rows:
        break

    pivot_row = matrix_rank
    while pivot_row < n_rows and abs(reduced_data[pivot_row, col]) < tolerance:
        pivot_row += 1

    if pivot_row == n_rows:
        continue

    reduced_data[[matrix_rank, pivot_row]] = reduced_data[[pivot_row, matrix_rank]]
    reduced_data[matrix_rank] = reduced_data[matrix_rank] / reduced_data[matrix_rank, col]

    for next_row in range(matrix_rank + 1, n_rows):
        mult = reduced_data[next_row, col]
        reduced_data[next_row] = reduced_data[next_row] - mult * reduced_data[matrix_rank]
    matrix_rank += 1

print("Rank of dataset:", matrix_rank)

In [ ]:
def matrix_transpose(mat):
    rows = len(mat)
    cols = len(mat[0])
    transposed = np.zeros((cols, rows))
    for row in range(rows):
        for col in range(cols):
            transposed[col, row] = mat[row, col]
    return transposed


def mat_mult(mat1, mat2):
    r1 = len(mat1)
    c1 = len(mat1[0])
    r2 = len(mat2)
    c2 = len(mat2[0])

    if c1 != r2:
        raise ValueError("The matrices cannot be multiplied: inner dimensions differ.")

    result = np.zeros((r1, c2))
    for row in range(r1):
        for col in range(c2):
            for k in range(c1):
                result[row, col] += (
                    mat1[row, k]
                    * mat2[k, col]
                )
    return result


n_samples = dataset.shape[0]
data_transpose = matrix_transpose(dataset)
gram_matrix = mat_mult(data_transpose, dataset) / n_samples

print("n =", n_samples)
print("Shape of data^T =", data_transpose.shape)
print("Gram matrix =")
print(gram_matrix)

In [ ]:
def euclidean_norm(vector):
    flat_vec = vector.reshape(-1)
    sum_sq = 0.0
    for val in flat_vec:
        sum_sq += val * val
    return np.sqrt(sum_sq)


def eigenvalue_power_iteration(matrix, tolerance=1e-8, max_iter=10000):
    vec = np.ones((matrix.shape[0], 1))
    vec = vec / euclidean_norm(vec)
    old_eval = 0

    for iteration in range(1, max_iter + 1):
        new_vec = mat_mult(matrix, vec)
        new_len = euclidean_norm(new_vec)
        if new_len == 0:
            raise ValueError("The power method reached a zero vector.")
        vec = new_vec / new_len

        vec_t = matrix_transpose(vec)
        mat_vec = mat_mult(matrix, vec)
        rayleigh = mat_mult(vec_t, mat_vec)
        eval_est = rayleigh[0, 0]

        if abs(eval_est - old_eval) < tolerance:
            return eval_est, vec.reshape(matrix.shape[0]), iteration
        old_eval = eval_est

    return eval_est, vec.reshape(matrix.shape[0]), max_iter


primary_eigenval, primary_eigenvec, iter_count_1 = eigenvalue_power_iteration(gram_matrix)

print("Largest eigenvalue:", primary_eigenval)
print("Corresponding eigenvector:")
print(primary_eigenvec)
print("Number of iterations:", iter_count_1)

In [ ]:
vec_col = primary_eigenvec.reshape(gram_matrix.shape[0], 1)
vec_t = matrix_transpose(vec_col)
outer_prod = mat_mult(vec_col, vec_t)
proj_times_gram = mat_mult(outer_prod, gram_matrix)
deflated_cov = gram_matrix - proj_times_gram
secondary_eigenval, secondary_eigenvec, iter_count_2 = eigenvalue_power_iteration(deflated_cov)

print("Second eigenvalue:", secondary_eigenval)
print("Corresponding eigenvector:")
print(secondary_eigenvec)
print("Number of iterations:", iter_count_2)

In [ ]:
computed_evals, computed_evecs = np.linalg.eigh(gram_matrix)

ordering = np.argsort(computed_evals)[::-1]
computed_evals = computed_evals[ordering]
computed_evecs = computed_evecs[:, ordering]

print("Eigenvalues from NumPy (largest to smallest):")
print(computed_evals)
print("\nEigenvectors from NumPy (columns):")
print(computed_evecs)
print("\nComparison with the power method:")
print("Power method lambda_1:", primary_eigenval)
print("NumPy lambda_1:", computed_evals[0])
print("Power method lambda_2:", secondary_eigenval)
print("NumPy lambda_2:", computed_evals[1])
print("Note: Eigenvectors may differ by sign as both v and -v are equivalent.")

In [ ]:
def count_convergence_steps(matrix, ref_eigenval, tolerance=1e-8):
    vec = np.ones((matrix.shape[0], 1))
    vec = vec / euclidean_norm(vec)

    for iteration in range(1, 10000):
        new_vec = mat_mult(matrix, vec)
        new_len = euclidean_norm(new_vec)
        if new_len == 0:
            raise ValueError("The power method reached a zero vector.")
        vec = new_vec / new_len

        vec_t = matrix_transpose(vec)
        mat_vec = mat_mult(matrix, vec)
        rayleigh = mat_mult(vec_t, mat_vec)
        approx_eval = rayleigh[0, 0]
        error = abs(approx_eval - ref_eigenval)

        if error < tolerance:
            return iteration

    return 10000


iter_lambda_1 = count_convergence_steps(gram_matrix, computed_evals[0])
iter_lambda_2 = count_convergence_steps(deflated_cov, computed_evals[1])

print("Required accuracy:", 1e-8)
print("Iterations for lambda_1:", iter_lambda_1)
print("Iterations for lambda_2:", iter_lambda_2)
print("Convergence speed depends on the ratio of eigenvalues.")